# Phase 9 — Risk Score Generation
**Problem Statement ID 26184:** Predictive Analytics Framework for Cybercrime Complaints to Forecast Likely Cash Withdrawal Locations

## Objective
Convert trained XGBoost predicted probabilities of future cash withdrawal into:
1. Probability score
2. 0–100 integer Risk Score
3. Operational risk categories (`LOW`, `MODERATE`, `HIGH`, `CRITICAL`)
4. Human-in-the-loop operational interpretations
5. Aggregated location-level risk summaries

**Core Governance:** The risk score represents predicted statistical likelihood of a qualifying withdrawal within 24h, NOT proof of criminal guilt.


## 1. Load Inputs
Load predictions, processed test dataset, and metadata.

In [ ]:
import json
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

warnings.filterwarnings('ignore')

BASE_DIR = Path('..').resolve()
DATA_DIR = BASE_DIR / 'data' / 'processed'
OUTPUTS_DIR = BASE_DIR / 'outputs'
FIG_DIR = OUTPUTS_DIR / 'figures'
PRED_DIR = OUTPUTS_DIR / 'predictions'

input_profile = pd.read_csv(OUTPUTS_DIR / 'phase9_input_profile.csv')
display(input_profile)

preds_df = pd.read_csv(PRED_DIR / 'xgboost_test_predictions.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
print(f"Loaded predictions: {preds_df.shape}")
print(f"Loaded test dataset: {test_df.shape}")


## 2. Inspect Probabilities
Examine the distribution and characteristics of `probability_future_withdrawal`.

In [ ]:
proba = preds_df['probability_future_withdrawal']
print("Probability Summary:")
display(proba.describe())


## 3. Validate Probabilities
Ensure probabilities are numeric, non-null, finite, and strictly bounded within [0, 1].

In [ ]:
prob_val = pd.read_csv(OUTPUTS_DIR / 'phase9_probability_validation.csv')
display(prob_val)


## 4. Generate Risk Scores
Formula:
$$\text{risk\_score} = \text{round}(\text{probability} \times 100)$$

In [ ]:
def probability_to_risk_score(probability):
    return np.round(np.clip(probability, 0.0, 1.0) * 100.0).astype(int)

scores = probability_to_risk_score(proba)
print(f"Generated {len(scores)} risk scores. Min: {scores.min()}, Max: {scores.max()}, Mean: {scores.mean():.2f}")


## 5. Assign Operational Categories & Interpretations
Categories:
- `0–39`: LOW
- `40–59`: MODERATE
- `60–79`: HIGH
- `80–100`: CRITICAL

In [ ]:
def assign_risk_category(score):
    if score <= 39: return 'LOW'
    elif score <= 59: return 'MODERATE'
    elif score <= 79: return 'HIGH'
    else: return 'CRITICAL'

risk_scores_df = pd.read_csv(OUTPUTS_DIR / 'phase9_risk_scores.csv')
display(risk_scores_df.head(10))


## 6. Score Statistics
Examine summary statistics of predicted probabilities and risk scores.

In [ ]:
score_stats = pd.read_csv(OUTPUTS_DIR / 'phase9_risk_score_statistics.csv')
display(score_stats)


## 7. Category Distribution
Inspect frequency and proportions across the four operational tiers.

In [ ]:
cat_dist = pd.read_csv(OUTPUTS_DIR / 'phase9_risk_category_distribution.csv')
display(cat_dist)


## 8. Diagnostic Threshold Analysis
Evaluate operational risk thresholds (20, 30, 40, 50, 60, 70, 80).
*Note: Diagnostic only — not used for model selection or tuning.*

In [ ]:
thresh_report = pd.read_csv(OUTPUTS_DIR / 'phase9_risk_threshold_report.csv')
display(thresh_report)


## 9. Location-Level Risk Summary
Aggregated complaints, mean probabilities, and dominant risk tiers across victim districts.

In [ ]:
loc_summary = pd.read_csv(OUTPUTS_DIR / 'phase9_location_risk_summary.csv')
print(f"Total location groups analyzed: {len(loc_summary)}")
display(loc_summary.head(15))

high_summary = pd.read_csv(OUTPUTS_DIR / 'phase9_high_risk_summary.csv')
print("High-Risk Category Summary:")
display(high_summary)


## 10. Quality Validation & Monotonicity
Check score constraints, boundary unit tests, and category monotonicity.

In [ ]:
val_report = pd.read_csv(OUTPUTS_DIR / 'phase9_risk_validation_report.csv')
display(val_report)

mono_report = pd.read_csv(OUTPUTS_DIR / 'phase9_monotonicity_report.csv')
display(mono_report)

outcome_val = pd.read_csv(OUTPUTS_DIR / 'phase9_risk_category_outcome_validation.csv')
print("Risk Category vs Actual Outcome Validation:")
display(outcome_val)


## 11. Visualizations
Visual representation of score distribution and category breakdown.

In [ ]:
print("Risk Score Distribution (0-100):")
display(Image(filename=str(FIG_DIR / 'phase9_risk_score_distribution.png')))

print("Operational Risk Category Breakdown:")
display(Image(filename=str(FIG_DIR / 'phase9_risk_category_chart.png')))


## 12. Conclusions & Phase 10 Readiness
1. **Risk Score Conversion:** Probability outputs smoothly map to the 0–100 integer risk scale and four operational categories.
2. **Monotonic Ordering:** Mean probability monotonically increases across categories ($0.2975 \to 0.4460 \to 0.6104$).
3. **High-Risk Prioritization:** High-risk complaints showed double the baseline empirical withdrawal rate ($20.00\%$ vs $10.33\%$).
4. **Human-in-the-Loop:** Decisions provide triage intelligence for authorized law enforcement analysts.

**Status:** Phase 9 Complete. Ready for Phase 10 — Spatial Hotspot & Geographic Cluster Analysis.
